In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
sns.set_style('whitegrid')
sns.set_palette("Set2")

%matplotlib inline

# Leer los datos

In [2]:
totales = pd.read_csv("../../../data/respuestas_fede.csv")
print(totales.shape)

#globales
marmol = totales.loc[totales["escuela"] == "Colegio Modelo Mármol"]
mantovani = totales.loc[totales["escuela"] == "Escuela Nueva Juan Mantovani"]

file_ext = '.png'
dpi_value = 200

(369, 22)


## Qué es la nube vs Funcionalidad de la nube

In [15]:
# funcion aux para analizar misc
def analizar_misc_como_viaja_wpp(val):
    if val == "El mensaje se manda a través de una antena.":
        return "Misc. débil"
    if val == "El mensaje se manda directamente.":
        return "Misc. fuerte"
    if val == "El mensaje se manda a través de un satélite.":
        return "Misc. fuerte"
    if val == "El mensaje se manda a través de la nube.":
        return "Misc. fuerte"
    if val == "El mensaje se manda a través de una red de antenas y computadoras.":
        return "Sin Misc."
    if val == "No sé":
        return "No sé"

def analizar_misc_que_es_nube(val):
    if val == "Una nube":
        return "Misc. fuerte"
    if val == "Una parte del celular":
        return "Misc. fuerte"
    if val == "Una computadora gigante":
        return "Misc. fuerte"
    if val == "Una red de antenas y satélites":
        return "Misc. débil"
    if val == "Muchísimas computadoras (tantas que podrían llenar una cancha de fútbol)":
        return "Sin Misc."
    if val == "No sé":
        return "No sé"
    

In [21]:
# Boilerplate para generar grafico de nube vs func de nube
def nube_vs_como_viaja(df, df_name, agrupando = False, invertir_porcentajes = False):
    que_es_nube = df["que_es_nube"]
    como_viaja_wpp = df["como_viaja_wpp"]

    if(agrupando):
        labels_que_es_nube = [
                            "Misc. fuerte",
                            "Misc. débil",
                            "No sé",
                            "Sin Misc.",
                            ]
        
        labels_como_viaja_wpp = [
                            "Sin Misc.",
                            "No sé",
                            "Misc. débil",
                            "Misc. fuerte",
                            ]
        
        que_es_nube = que_es_nube.apply(analizar_misc_que_es_nube)
        como_viaja_wpp = como_viaja_wpp.apply(analizar_misc_como_viaja_wpp)

        if(not invertir_porcentajes):
            filas    = labels_que_es_nube
            columnas = labels_como_viaja_wpp

            cross_tab_result = pd.crosstab(que_es_nube, como_viaja_wpp)
        else:
            filas    = labels_como_viaja_wpp
            filas.reverse()
            columnas = labels_que_es_nube
            columnas.reverse()

            cross_tab_result = pd.crosstab(como_viaja_wpp, que_es_nube)
        
    else:
        # mas misc primero
        labels_que_es_nube = [
                 "Una nube",
                 "Una parte\ndel celu",
                 "Una compu\ngigante",
                 "Una red\nde antenas\ny satélites",
                 "No sé",
                 "Muchísimas\ncompus",
                ]
        # menos misc primero
        labels_como_viaja_wpp = [
                                 "Red de\nantenas y\ncompus",
                                 "No sé",
                                 "Una\nantena",
                                 # estas me parecen todas fuertes
                                 "Directamente",
                                 "Un\nsatélite",
                                 "La nube",
                                 ]

        rename_que_es_nube = {"Muchísimas computadoras (tantas que podrían llenar una cancha de fútbol)": "Muchísimas\ncompus",
                                "Una computadora gigante":"Una compu\ngigante",
                                "Una parte del celular": "Una parte\ndel celu",
                                "Una red de antenas y satélites": "Una red\nde antenas\ny satélites"}
        
        rename_como_viaja_wpp = {"El mensaje se manda a través de una antena.": "Una\nantena",
                              "El mensaje se manda directamente.": "Directamente",
                              "El mensaje se manda a través de un satélite.": "Un\nsatélite",
                              "El mensaje se manda a través de una red de antenas y computadoras.": "Red de\nantenas y\ncompus",
                              "El mensaje se manda a través de la nube.": "La nube"}

        
        if(not invertir_porcentajes):
            filas = labels_que_es_nube
            columnas = labels_como_viaja_wpp
            cross_tab_result = pd.crosstab(que_es_nube, como_viaja_wpp)
            cross_tab_result = cross_tab_result.rename(index=rename_que_es_nube, columns=rename_como_viaja_wpp)
        else:
            filas = labels_como_viaja_wpp
            filas.reverse()
            columnas = labels_que_es_nube
            columnas.reverse()
            cross_tab_result = pd.crosstab(como_viaja_wpp, que_es_nube)
            cross_tab_result = cross_tab_result.rename(index=rename_como_viaja_wpp, columns=rename_que_es_nube)


    cross_tab_result = cross_tab_result.reindex(filas, columns=columnas)

    def agregar_totales(fila):
       res = 0
       for i in list(range(0,fila.size)):
            res = res + fila[i]
       return res
    
    def dividir(fila):
       return fila.div(fila[fila.size - 1])

    cross_tab_result['totales'] = cross_tab_result.apply(agregar_totales, axis=1)
    cross_tab_result = cross_tab_result.apply(dividir, axis=1)
    cross_tab_result = cross_tab_result.drop(['totales'], axis=1)

    titulo_que_es_nube = '¿Que es la nube?'
    titulo_como_viaja_wpp = 'Cómo viaja un mensaje de wpp sin wifi'

    if(not invertir_porcentajes):
        titulo_filas = titulo_que_es_nube
        titulo_columnas = titulo_como_viaja_wpp
    else:
        titulo_filas = titulo_como_viaja_wpp
        titulo_columnas = titulo_que_es_nube

    sns.heatmap(cross_tab_result, cmap='YlGnBu', annot=True, fmt='.2f', linewidths=0.5, vmin=0.0, vmax=1.0)
    plt.title('Relación de Respuestas\nQue es la nube vs Cómo viaja un mensaje de wpp sin wifi\nDatos {} - Agrupando {} - Invirtiendo {}'.format(df_name,agrupando,invertir_porcentajes))
    plt.xlabel(titulo_columnas)
    plt.xticks(rotation=0)
    plt.ylabel(titulo_filas)
    plt.yticks(rotation=0)
    # plt.show()
    plt.savefig('nube_vs_como_viaja_wpp_{}_agrupando_{}_invirtiendo_{}'.format(df_name,agrupando,invertir_porcentajes)+file_ext, bbox_inches='tight', dpi=dpi_value)
    plt.clf()

In [22]:
# llamo al generador de graficos con los tres dfs
nube_vs_como_viaja(totales, "totales")
nube_vs_como_viaja(marmol, "marmol")
nube_vs_como_viaja(mantovani, "mantovani")

# Agrupando misc
nube_vs_como_viaja(totales,   "totales", True)
nube_vs_como_viaja(marmol,    "marmol", True)
nube_vs_como_viaja(mantovani, "mantovani", True)

# Invirtiendo
nube_vs_como_viaja(totales,   "totales",   invertir_porcentajes=True)
nube_vs_como_viaja(marmol,    "marmol",    invertir_porcentajes=True)
nube_vs_como_viaja(mantovani, "mantovani", invertir_porcentajes=True)

# Agrupando misc e Invirtiendo
nube_vs_como_viaja(totales,   "totales",  agrupando=True, invertir_porcentajes=True)
nube_vs_como_viaja(marmol,    "marmol",   agrupando=True, invertir_porcentajes=True)
nube_vs_como_viaja(mantovani, "mantovani",agrupando=True, invertir_porcentajes=True)


<Figure size 640x480 with 0 Axes>